In [ ]:
# Creston Getz 7/13/2026
# This is the main file for the Grazioso Salvare Dashboard.
# It is a Dash app that allows users to filter and view data from the Austin Animal Center database.
# Dash communicates with MongoDB using APIs/HTTP. app.py and api.py implement the backend of the app.


import dash
from dash import dcc, html, dash_table
import dash_auth
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import base64

from urllib.parse import quote_plus

import os
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import requests

###################################
# Database Connection and API Calls
###################################
API_URL = "http://127.0.0.1:5001"
load_dotenv() # Load env vars

COLUMN_ORDER = [
    "animal_id",
    "intake_date",
    "name",
    "age",
    "animal_type",
    "breed",
    "color",
    "sex_upon_outcome",
    "found_address",
    "age_in_weeks",
    "health_condition_at_intake",
    "date_of_birth",
    "source",
]

# The API data does not have any coordinate data.
# This method allows users to click on the address of an animal and go to Google Maps
# Inspired by: https://www.youtube.com/watch?v=d8dQgj77kpg&t=23s and examples from Claude
def add_map_link(df):
    if "found_address" not in df.columns:
        return df

    df["found_address"] = df["found_address"].apply(
        lambda address: address if address == "Unknown"
        # quote_plus helps protect URL
        else f"[{address}](https://www.google.com/maps/search/?api=1&query={quote_plus(address)})"
    )

    return df


# This will make an API call to the backend to get data from MongoDB and store it in a DataFrame.
# The rescue_type parameter is used to filter the animals by the rescue type for the company.
# If no rescue type is selected, it will return all animals or no filter.
def get_animals(rescue_type=None):
    if rescue_type and rescue_type != 'None':
        response = requests.get(f"{API_URL}/api/animals/filter", 
        params={'rescue_type': rescue_type},
        headers={'X-API-KEY': os.environ['API_KEY']})
    else:
        # If no rescue type, then return all animals
        response = requests.get(f"{API_URL}/api/animals", headers={'X-API-KEY': os.environ['API_KEY']})
    
    response.raise_for_status()
    df = pd.DataFrame(response.json())
    df = df.drop(columns=['_id'], errors='ignore') # Drop the MongoDB _id column

    # Reorder columns using the order list at the top of the file.
    # The list comprehension helps prevent key errors and dropping columns not in the list.
    ordered_columns = [column for column in COLUMN_ORDER if column in df.columns]
    df = df[ordered_columns + [column for column in df.columns if column not in ordered_columns]]

    # Sort the DataFrame so the newest animals are shown first
    if 'intake_date' in df.columns:
        df = df.sort_values("intake_date", ascending=False, na_position="last", 
                            key= lambda s: pd.to_datetime(s, errors="coerce"),
        )

    # Convert null dates to an Unknown string for readability
    df["date_of_birth"] = df["date_of_birth"].fillna("Unknown")

    return add_map_link(df)

# Loads data into df. Empty if error
try:
    df = get_animals()
except Exception as e:
    print(f"Error with API call: {e}")
    df = pd.DataFrame() # Return an empty DataFrame if the API fails



#########################
# Dashboard Layout / View
#########################
app = dash.Dash(__name__)
app.server.secret_key = os.environ['FLASK_SECRET_KEY']

# This code was inspired by https://community.plotly.com/t/flask-authentication-with-dash-pages/62958 used 7/13/26 and examples from Claude.
# Because this file is in a Jupyter notebook, the app will not start if the env password is not set.
# It prevents the app from failing to start due to a missing env var, and locks the app with a password and username.
dash_password = os.environ.get('DASH_PASSWORD')
if not dash_password:
    raise RuntimeError(
        "DASH_PASSWORD is not set. The dashboard will not start without authentication."
    )

dash_auth.BasicAuth(app, {'admin': dash_password}, public_routes=["/_alive_<token>"],)


# Grazioso Salvare Logo
image_filename = 'resources/Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# The app.layout section creates the HTML code for the dashboard.
app.layout = html.Div([
    # Style and create the header with the logo on the left
    html.Div(
    style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center',
           'backgroundColor': '#D3D3D3', 'padding': '15px', 'borderRadius': '8px'},
    children=[
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()), # Adds and decodes the logo
            alt="Small picture of a red dog, which is the Grazioso Salvare logo, next to the title",
            style={'width': '100px', 'height': 'auto', 'marginRight': '20px'}
        ),
        html.Div(
            style={'textAlign': 'center'},
            children=[
            html.H1('Grazioso Salvare Dashboard', style={'margin': '0'}),
            ]
        )
    ]),
    html.Hr(),
    # Section for the interactive filtering options. Adds a dropdown menu.
    html.Div([
        dcc.Dropdown(['None', 'Water Rescue', 'Mountain or Wilderness Rescue', 
                      'Disaster Rescue or Individual Tracking'], value = 'None',
                     id='filter-type', searchable=False, placeholder="Select a Filter",),
        html.P("Select Address Link to Load Google Maps", 
           style={'fontWeight': 'bold', 'color': '#000000', 'marginBottom': '5px'}),

        # Outputs an error message if there is a problem with the dashboard.
        html.Div(id='error-message', style={'color': 'red', 'fontWeight': 'bold', 'marginTop': '10px'})

    ]),
    html.Hr(),
    # Create the main Datatable
    # dcc.Loading adds a loading circle to improve UI and UX if the response is slow.
    # In testing, all responses were very fast
    dcc.Loading(dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True,
                                   "presentation": "markdown" if i == "found_address" else "input"} for i in df.columns],
                         data=df.to_dict('records'),
        # Features for the interactive data table to make it user-friendly for the client

        # Adds a radio button on the side of the table
        # row_selectable = "single",
        # selected_rows=[0],

        # Puts a radio button in each column header so the user can select one column.
        # This is what actually populates selected_columns for the highlight callback.
        column_selectable='single',
        selected_columns=[],

        style_header={'fontWeight': 'bold',  # Make the header clearer
                        'text-transform': 'uppercase',
                        'backgroundColor': '#D3D3D3',
                        'color': '#000000'
                      },
        
        style_data={'borderColor': '#333'},

        style_table={'overflowX': 'auto'}, # Add a scroll bar
        style_cell={'textAlign': 'left', # Left align text and add padding
                        'padding_right': '20px', 

                    }, 
        
        # Enable pagination
        page_action='native',
        page_current=0,
        page_size=10,
        
        # Add sorting and filtering to columns
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        filter_options={"placeholder_text": "Search column..."},

        # Make links open in a new tab
        markdown_options={"link_target": "_blank"},


    ), type="circle"),
    
    html.Br(),
    html.Hr(),
    # This sets up the dashboard so pie chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex', 'marginBottom':'50px'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
# Updates the datatable and chart based on the filter the user selects. Each time it is called, it will call the get_animals method.
@app.callback(
    [Output('datatable-id', 'data'),
    Output('datatable-id', 'columns'),
    Output('error-message', 'children')],
    [Input('filter-type', 'value')]
)
# This method was inspired by 
# https://dash.plotly.com/callback-error-handlers accessed 7/14/26
# https://dash.plotly.com/basic-callbacks accessed 7/14/26
# And a few examples from Claude.
# I was having trouble handling the errors when the user would select different filters.
# This method is the main callback for the application.
def update_dashboard(filter_type):
    try:
        animals = get_animals(filter_type)
        # Columns built here help with load failures
        columns = [{"name": i, "id": i, "deletable": False, "selectable": True, "presentation": "markdown" if i == "found_address" else "input"}
                   for i in animals.columns]
        records = animals.to_dict('records')
        return records, columns, '' # Resets the selection when the data is swapped. records is always returned as is
    except requests.exceptions.RequestException as e:
        return [], [], html.Div(
            "Sorry the dashboard could not be loaded. Please try again.", style={'color': 'red', 'fontWeight': 'bold'}
        )
    except Exception:
        return [], [], html.Div(
            "An error occurred loading the dashboard. Try refreshing the page.", style={'color': 'red', 'fontWeight': 'bold'}
        )


# Display the breeds of animals based on the quantity represented in
# the data table. This method makes a pie chart based on whatever filter the user selects
# to show the distribution of dog breeds only
@app.callback(
     Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Return nothing if there is no data
    if not viewData:
        return []
    
    # Creates a DataFrame of only dogs and will return a p tag if no dog data is found
    dog_df = pd.DataFrame.from_dict(viewData)
    if 'animal_type' not in dog_df.columns:
        return [html.P("No dog data available for this filter.")]
    
    dog_df = dog_df[dog_df['animal_type'] == 'Dog']
    
    # Gets the top 10 breed value counts for dogs
    top_breeds = dog_df['breed'].value_counts().nlargest(10).reset_index()
    top_breeds.columns = ['breed', 'count']
    
    # Returns the figure
    return [
       dcc.Graph(            
           figure = px.pie(top_breeds, values='count', names='breed', title='Top 10 Dog Breed Distribution')
       )
    ]

    
# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D3D3D3'
    } for i in selected_columns]


# Run the app and display the result in JupyterLab mode. Note: if you have previously run a prior app,
# the default port of 8050 may not be available. If so, try setting an alternate port.
app.run(debug=True, port=8050, jupyter_mode='inline', mode='external') 

 http://127.0.0.1:8050